# スネークゲーム（p5.js ゲーム）

ヘビを操作してエサを食べ、できるだけ長くするゲームです。

## ルール
- エサ（赤い四角）を食べるとヘビが 1 マス長くなり **1 点**
- 壁か自分の体にぶつかるとゲームオーバー

## 操作
- **矢印キー**: ヘビの向きを変える（先にゲーム画面をクリックしてください）
- **スペースキー または クリック**: ゲームオーバー後にもう一度遊ぶ

## このノートブックの使い方

- コードセルを上から順番に **Shift + Enter** で実行し、最後の `%show` セルを実行するとゲーム画面が表示されます。
- キーボードで操作するゲームは、**最初にゲーム画面をクリック** してから操作してください（クリックでキー入力が画面に届くようになります）。
- コードを書き換えたら、そのセルを実行し直してから `%show` をもう一度実行すると、新しいゲームになります。
- 動かなくなったら、メニューの **Kernel → Restart Kernel and Clear Outputs of All Cells...** で最初からやり直せます。

p5.js の基本は `p5-tutorial.ipynb` で学べます。

## 1. ゲームの状態

画面を **20 × 20 のマス目** で考えます（1 マス = 20px）。ヘビは「マスの座標の配列」で表し、先頭が頭です。
向きは `dir` に `{x, y}` の形で持ちます（右なら `{x: 1, y: 0}`）。

In [ ]:
const CELL = 20;                  // 1 マスの大きさ（px）
const COLS = 20;                  // 横のマス数
const ROWS = 20;                  // 縦のマス数

let snake = [{ x: 10, y: 10 }];   // ヘビの体（先頭が頭）
let dir = { x: 1, y: 0 };         // 進む向き
let nextDir = { x: 1, y: 0 };     // 次のフレームで適用する向き（キー入力を一時的に保存）
let food = { x: 15, y: 10 };      // エサの位置
let score = 0;
let gameOver = false;

## 2. setup と draw

`frameRate(8)` で 1 秒に 8 回だけ `draw()` を呼び、ヘビを 1 マスずつ動かします。
動かすときは「新しい頭を先頭に追加し、末尾を削除」します。エサを食べたときだけ末尾を削除しないことで長くなります。

In [ ]:
function setup() {
  createCanvas(COLS * CELL, ROWS * CELL);
  frameRate(8);
  textFont("sans-serif");
  placeFood();
}

function draw() {
  background(30);
  drawGrid();

  if (!gameOver) {
    moveSnake();
  }

  // エサ
  fill(230, 60, 60);
  noStroke();
  rect(food.x * CELL, food.y * CELL, CELL, CELL, 4);

  // ヘビ（頭は明るい緑）
  for (let i = 0; i < snake.length; i++) {
    fill(i === 0 ? color(120, 255, 120) : color(60, 180, 60));
    rect(snake[i].x * CELL, snake[i].y * CELL, CELL, CELL, 4);
  }

  // スコア
  fill(255);
  textSize(16);
  textAlign(LEFT, TOP);
  text("スコア: " + score, 8, 6);

  if (gameOver) {
    fill(0, 170);
    rect(0, 0, width, height);
    fill(255);
    textAlign(CENTER, CENTER);
    textSize(32);
    text("ゲームオーバー", width / 2, height / 2 - 20);
    textSize(16);
    text("スコア: " + score + "　スペースキーかクリックでもう一度", width / 2, height / 2 + 25);
  }
}

function drawGrid() {
  stroke(45);
  for (let i = 0; i <= COLS; i++) line(i * CELL, 0, i * CELL, height);
  for (let j = 0; j <= ROWS; j++) line(0, j * CELL, width, j * CELL);
}

function moveSnake() {
  dir = nextDir;
  const head = { x: snake[0].x + dir.x, y: snake[0].y + dir.y };

  // 壁にぶつかった？
  if (head.x < 0 || head.x >= COLS || head.y < 0 || head.y >= ROWS) {
    gameOver = true;
    return;
  }
  // 自分の体にぶつかった？
  for (const part of snake) {
    if (part.x === head.x && part.y === head.y) {
      gameOver = true;
      return;
    }
  }

  snake.unshift(head);   // 新しい頭を先頭に追加

  if (head.x === food.x && head.y === food.y) {
    score++;             // エサを食べた: 末尾を残して長くする
    placeFood();
  } else {
    snake.pop();         // 食べていない: 末尾を削除して長さを保つ
  }
}

// ヘビの体と重ならない場所にエサを置く
function placeFood() {
  while (true) {
    const candidate = { x: floor(random(COLS)), y: floor(random(ROWS)) };
    const onSnake = snake.some((p) => p.x === candidate.x && p.y === candidate.y);
    if (!onSnake) {
      food = candidate;
      return;
    }
  }
}

## 3. 入力とリセット

矢印キーは `keyCode` と定数 `UP_ARROW` などで判定します。**逆方向への入力は無視** します（そのままだと自分の体にぶつかってしまうため）。
`keyPressed()` で `return false` すると、矢印キーでページがスクロールするのを防げます。

In [ ]:
function keyPressed() {
  if (keyCode === UP_ARROW && dir.y !== 1) nextDir = { x: 0, y: -1 };
  if (keyCode === DOWN_ARROW && dir.y !== -1) nextDir = { x: 0, y: 1 };
  if (keyCode === LEFT_ARROW && dir.x !== 1) nextDir = { x: -1, y: 0 };
  if (keyCode === RIGHT_ARROW && dir.x !== -1) nextDir = { x: 1, y: 0 };
  if (key === " " && gameOver) resetGame();
  return false;   // ページのスクロールなど、ブラウザの標準動作を止める
}

function mousePressed() {
  if (gameOver) resetGame();
}

function resetGame() {
  snake = [{ x: 10, y: 10 }];
  dir = { x: 1, y: 0 };
  nextDir = { x: 1, y: 0 };
  score = 0;
  gameOver = false;
  placeFood();
}

In [ ]:
%show 100% 410px

## 改造のヒント

- スコアが増えるたびに `frameRate()` を上げて、だんだん速くしてみましょう
- 壁にぶつからず反対側から出てくる「ワープ」仕様にしてみましょう（`head.x = (head.x + COLS) % COLS`）
- 毒エサ（食べると短くなる）や、時間で消えるエサを追加してみましょう
- ハイスコアを表示してみましょう